In [ ]:
from pathlib import Path

import numpy as np
import scipy.optimize as spo

from cardiac_electrophysiology import builder
from cardiac_electrophysiology.utils import visualization

In [ ]:
settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        vtu_mesh_path=Path("../data/patient_01/mesh_with_fibers_tags.vtu"),
        xdmf_mesh_path=Path("../data/patient_01/mesh.xdmf"),
        basis_vecs_path=Path("../data/patient_01/basis_vecs.npy"),
        log_file_path=Path("posterior_logfile.log"),
        noisy_data_path=None,
        fiber_ensemble_path=None,
    ),
    strategies=builder.Strategies(
        ground_truth_strategy="from_mesh",
        mean_strategy="from_ground_truth",
        noisy_data_strategy="from_ground_truth",
    ),
    prior_parameters=builder.PriorParameters(
        kappa=1.0,
        tau=1.0,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=100,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

In [ ]:
posterior_builder = builder.PosteriorBuilder(settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)

In [ ]:
optimizer_options = {
    "disp": True,
    "maxiter": 1000,
    "ftol": 1e-6,
    "gtol": 1e-6,
    "maxls": 100,
}

map_estimate = spo.minimize(
    fun=posterior.evaluate_cost,
    jac=posterior.evaluate_gradient,
    x0=additional_output.prior_mean_parameter,
    method="L-BFGS-B",
    options=optimizer_options,
)
np.save("map_estimate.npy", map_estimate.x)

In [ ]:
map_parameter = np.load("map_estimate.npy")
map_angles = additional_output.angle_transformator.compute_angle_from_parameter(map_parameter)
map_fibers = additional_output.fiber_transformator.compute_fiber_from_angle(map_angles)

In [ ]:
visualization.visualize_partial_scalar_field(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=map_angles,
)
visualization.visualize_full_vector_field(
    mesh=additional_output.pv_mesh,
    vector_field=map_fibers,
    scaling_factor=1,
)